# 🧬 JORINOVA NEXUS — Histopathology (tissue) detector (fine-tuning)

Detects benign/malignant tissue + tumour features. Pick an organ-specific dataset per case.

Produces `histology.pt`; detected classes resolve to their finding/organism via
`backend/ai_services/histology_findings.json`.

> **Fine-tuning, NOT from scratch** (starts from `yolov8s.pt`).
> **Keyless of Kaggle** — data from Roboflow (your free key), no GitHub token.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_histology', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_histology')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (Roboflow — no Kaggle, no GitHub token)

In [ ]:
# ── 4. Dataset (Roboflow, keyless of Kaggle) -> DATA_YAML ──
# CANDIDATES tried in order (auto-version). Add your own from
#   https://universe.roboflow.com/search?q=histology  (DETECTION project -> Download -> YOLOv8 -> "show download code").
import os, glob, yaml
from getpass import getpass
os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow Private API Key: ').strip()
try:
    from roboflow import Roboflow
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','roboflow'], check=True)
    from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])

CANDIDATES = [
    ('yolo-usyrb', 'lung-cancer-qmdnh-kps7i')
]
CANDIDATES = [(w, p) for (w, p) in CANDIDATES if 'WORKSPACE' not in w]
assert CANDIDATES, 'No dataset configured. Search https://universe.roboflow.com/search?q=histology and add a (workspace, project).'
ds = None
for w, p in CANDIDATES:
    try:
        proj = rf.workspace(w).project(p)
        for v in range(1, 12):
            try:
                ds = proj.version(v).download('yolov8', location='/content/data/histology_rf')
                print('OK', w + '/' + p, 'v', v); break
            except Exception:
                continue
        if ds: break
    except Exception as e:
        print('skip', w + '/' + p, '-', str(e)[:70])
assert ds, 'None of the CANDIDATES downloaded — add another project from the search URL.'
DATA_YAML = os.path.join(ds.location, 'data.yaml')
print('DATA_YAML =', DATA_YAML, '| classes:', yaml.safe_load(open(DATA_YAML))['names'])
print('NOTE: if the dataset class names differ from the app keys, tell me the printed')
print('      names and I add aka-aliases to the histology map (or rename here).')

## 5. Choose base weights (fine-tune, never from scratch)

In [ ]:
BASE_WEIGHTS = 'yolov8s.pt'   # yolov8m.pt = more accurate/slower
print('fine-tuning from:', BASE_WEIGHTS)

## 6. Fine-tune the detector

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_histology/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/detect'
model = YOLO(BASE_WEIGHTS)
model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
            project=PROJECT, name='histology', pretrained=True, patience=25, plots=True,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, flipud=0.5, mosaic=1.0, translate=0.1, scale=0.5)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 6b. RESUME if a run disconnects (run INSTEAD of step 6)

In [ ]:
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_histology/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_histology/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_YAML' in globals(), 'Re-run step 4 first so DATA_YAML is set.'
    model = YOLO(ckpt); model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
        project='/content/drive/MyDrive/nexus_histology/runs', name='histology_cont', pretrained=True, patience=25)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = '/content/drive/MyDrive/nexus_histology'
best = sorted([c for c in glob.glob('/content/**/weights/best.pt', recursive=True)
               + glob.glob(DRIVE + '/**/weights/best.pt', recursive=True) if os.path.isfile(c)],
              key=os.path.getmtime)[-1]
print('using weights:', best)
from ultralytics import YOLO; print('classes:', YOLO(best).model.names)
try: os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/histology.pt'); print('Drive ->', DRIVE + '/histology.pt')
except Exception as e: print('drive copy skipped:', e)
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`histology.pt`** at **`backend/models/histology/histology.pt`**, commit, push.
2. Auto-loads for `image_type == "histology"`; classes resolve via `backend/ai_services/histology_findings.json`.
3. On Render, build with `INSTALL_ML=1` (>=2 GB); else Claude vision reads the field.